# WOJAN AI FACTORY — Ollama na GPU Kaggle (T4 ×2)

Uruchamia **lokalny serwer Ollama** na instancji Kaggle z 2 kartami **Tesla T4 (16 GB każda)**.

Ważne zasady:
- 2× T4 to **DWA osobne urządzenia po ~16 GB VRAM** — NIE jeden wspólny 32 GB pool. Każdy model działa na JEDNEJ karcie (chyba że użyjesz podziału warstw — Ollama domyślnie trzyma model na jednym GPU).
- Port Ollama (11434) **nigdy nie jest wystawiany publicznie**. Ruch idzie przez **Cloudflare Tunnel** z tokenem — tylko Twój komputer/telefon z tokenem ma dostęp.
- Sesje Kaggle bywają zamykane — notebook wykrywa utratę sesji i przypomina o rekonfiguracji.

Po uruchomieniu skopiuj **secured URL** i wklej go w `wojan-config.json` → `ollama.base` + `ollama.apiKey`.

In [ ]:
# 1) Sprawdzenie GPU — DWA osobne T4, NIE jeden pool 32 GB
!nvidia-smi
import subprocess
out = subprocess.check_output(['nvidia-smi', '--query-gpu=index,name,memory.total,memory.used', '--format=csv']).decode()
print(out)
lines = [l for l in out.strip().splitlines()[1:]]
print(f'Wykryte GPU: {len(lines)}')
for l in lines:
    print('  ', l.strip())
    mem = l.split(',')[2].strip()
    assert 'MiB' in mem or 'GiB' in mem, mem
print('OK — sprawdzono osobno dla każdej karty (brak wspólnego poola VRAM).')

In [ ]:
# 2) Instalacja Ollama (oficjalny skrypt) + model startowy
!curl -fsSL https://ollama.com/install.sh | sh
import os, time
os.environ['OLLAMA_HOST'] = '127.0.0.1:11434'   # TYLKO localhost — publicznie idzie tunel z tokenem
os.environ['OLLAMA_MODELS'] = '/kaggle/working/ollama-models'
!ollama serve > /kaggle/working/ollama.log 2>&1 &
time.sleep(4)
!ollama pull llama3.1:8b
!curl -s http://127.0.0.1:11434/api/version

In [ ]:
# 3) Kontrola VRAM PO załadowaniu modelu — która karta ile zajmuje
!nvidia-smi --query-gpu=index,name,memory.used,memory.total --format=csv
print('Model mieszka na jednej karcie; druga pozostaje wolna dla drugiego modelu/zadań.')
print('Nie sumuj VRAM — 2x T4 to 2x ~16 GB osobno, nie 32 GB łącznie.')

In [ ]:
# 4) Bezpieczny endpoint: Cloudflare Tunnel + token — NIGDY goły port 11434
import secrets, threading, subprocess, time, json
TOKEN = secrets.token_urlsafe(32)
with open('/kaggle/working/wojan-tunnel-token.txt', 'w') as f:
    f.write(TOKEN)
print('Token API (wpisz go w wojan-config.json -> ollama.apiKey):')
print(TOKEN)

# mały proxy: publiczny URL -> 127.0.0.1:11434, wymaga Authorization: Bearer <TOKEN>
proxy = r'''
import http.server, urllib.request, sys
TOKEN = sys.argv[1]
class H(http.server.BaseHTTPRequestHandler):
    def do_POST(self):
        auth = self.headers.get('Authorization','')
        if auth != 'Bearer ' + TOKEN:
            self.send_response(401); self.end_headers(); return
        n = int(self.headers.get('Content-Length', 0))
        body = self.rfile.read(n)
        req = urllib.request.Request('http://127.0.0.1:11434'+self.path, data=body, method='POST')
        req.add_header('Content-Type','application/json')
        try:
            r = urllib.request.urlopen(req, timeout=600)
            data = r.read()
            self.send_response(200)
            self.send_header('Content-Type','application/json')
            self.send_header('Content-Length', str(len(data)))
            self.end_headers()
            self.wfile.write(data)
        except Exception as e:
            self.send_response(502); self.end_headers()
    def do_GET(self):
        auth = self.headers.get('Authorization','')
        if auth != 'Bearer ' + TOKEN:
            self.send_response(401); self.end_headers(); return
        try:
            r = urllib.request.urlopen('http://127.0.0.1:11434'+self.path, timeout=60)
            data = r.read()
            self.send_response(200)
            self.send_header('Content-Type','application/json')
            self.send_header('Content-Length', str(len(data)))
            self.end_headers()
            self.wfile.write(data)
        except Exception as e:
            self.send_response(502); self.end_headers()
    def log_message(self, *a): pass
http.server.ThreadingHTTPServer(('127.0.0.1', 8766), H).serve_forever()
'''
with open('/kaggle/working/proxy.py','w') as f:
    f.write(proxy)
subprocess.Popen(['python3','/kaggle/working/proxy.py', TOKEN])
print('Proxy z tokenem na 127.0.0.1:8766')

In [ ]:
# 5) Cloudflare Tunnel (szybki tryb tryminalny) — skopiuj publiczny URL z wyniku
!curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /kaggle/working/cloudflared || echo 'Pobierz cloudflared ręcznie'
!chmod +x /kaggle/working/cloudflared
print('Uruchom poniżej i skopiuj https://...trycloudflare.com — to jest Twój secured endpoint.')
!/kaggle/working/cloudflared tunnel --url http://127.0.0.1:8766 2>&1 | head -40 &
time.sleep(8)
print('--- wklej URL + token do wojan-config.json: {"ollama": {"base": "https://...trycloudflare.com", "apiKey": "<TOKEN>"}} ---')

In [ ]:
# 6) Wykrywanie utraty sesji Kaggle + kontrola usługi
import urllib.request, time, datetime
def check():
    try:
        r = urllib.request.urlopen('http://127.0.0.1:11434/api/version', timeout=5)
        return 'OK', r.read().decode()
    except Exception as e:
        return 'STRACONO SESJĘ / Ollama nie działa', str(e)
for i in range(3):
    st, info = check()
    print(datetime.datetime.now(), st, info[:120])
    if 'STRACONO' in st:
        print('>>> Sesja Kaggle wygasła lub proces umarł. Uruchom ponownie komórki 2-5. '
              'Stan projektów nie ginie — jest na Twoim PC w data/store.json.')
    time.sleep(2)